<a href="https://colab.research.google.com/github/fdavenport/CIVE480A-Fall26/blob/main/Lab4_Extreme_Fire_Weather.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lab 4: Analyzing Extreme Fire Weather

In this lab, we will use python to:
* Calculate percentiles of a distribution
* Compare distributions using histograms (bar charts)
* Analyze changes in extreme fire weather

**Setting up**

Import libraries you may want to use for the assignment:

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

**Background**

You are part of a wildfire mitigation planning team for Fort Collins, CO. Your team is looking into how often extreme fire risk is occurring in the foothills west of the City. You are interested in understanding how wildfire risk is changing. Ultimately, this information could be used to determine when to implement restrictions on activities that could start a fire, and when to pre-position firefighting crews and equipment, and how much to budget for wildfire mitigation. 

It is very hard to predict when exactly a wildfire will occur, because wildfire ignition is caused by somewhat random occurrences of lightning or human activity. Instead, scientists often measure wildfire risk using a metric called the Canadian Forest Fire Weather Index. The Fire Weather Index, or "FWI", can be calculated each day based on the temperature, the relative humidity of the atmosphere, the wind speed, recent precipitation, and the dryness of the vegetation (in other words, the fuel for a potential fire). If a fire starts when the Fire Weather Index is high, these conditions mean the fire is much more likely to grow quickly and spread.

Your planning team has defined an "extreme" fire weather day as one of the worst 0.5% of days in the historical record. More specifically, for your analysis, an extreme fire weather day exceeds the 99.5th percentile of historical days. You will use the time period between 1970-2004 as a historical baseline to calculate this threshold.  

**Data**

Data for daily Fire Weather Index values in the foothills adjacent to Fort Collins, CO, is located on the course github. This data covers 1970-2025, and the scientific research paper describing the original data can be found [here](https://www.nature.com/articles/s41597-020-0554-z). 

The data is stored in a .csv file at the following url:

In [5]:
fwi_url = "https://raw.githubusercontent.com/fdavenport/CIVE480A-Fall26/refs/heads/main/data/fort_collins_fire_weather_index.csv"

The next cell of code reads the data as a pandas DataFrame, converts the dates to datetime format, and adds a column with the year of each daily value:

In [ ]:
fwi = pd.read_csv(fwi_url, index_col = 0)
fwi["date"] = pd.to_datetime(fwi["date"])
fwi["year"] = fwi["date"].dt.year
fwi.head()

**Splitting the data into a baseline period and a recent period**

You will compare extreme fire weather in the *baseline* period (the years before 2005, i.e. 1970-2004) to the *recent* period (2005 and later). 

The next cell shows how to create a subset of the data for the baseline period by selecting the rows where the year is less than 2005:

In [ ]:
fwi_early = fwi.loc[fwi["year"] < 2005]
fwi_early.tail(n = 3)

<font color="#0000FF">**1a)** Create a subset of the data for the recent period (2005 and later) called `fwi_recent`.
</font>

In [ ]:
## add code here


**Calculating a percentile**

A percentile tells us the value below which a given percentage of the data falls. For example, the 50th percentile (the median) is the value that is higher than half of the days and lower than the other half. You can calculate percentiles with the numpy `np.quantile` function (the terms quantile and percentile mean the same thing). The first argument to the `np.quantile` function is the data, and the second argument is the percentile expressed as a fraction between 0 and 1 (so the 50th percentile is 0.5).

The next cell calculates the 50th percentile of FWI in the baseline period:

In [ ]:
fwi_early_p50 = np.quantile(fwi_early["fwi"], 0.5)
print("50th percentile of FWI in the baseline period:", fwi_early_p50)

**Plotting a distribution as a bar chart**

A bar chart is a good way to show the *distribution* of a variable: we divide the range of FWI values into "bins" and show how many days fall into each bin. This takes two steps:
1. Use the numpy `np.histogram` function to count the number of days in each bin. It needs the data and the bin edges, and returns the counts.
2. Use `ax.bar` to plot one bar for each bin. The bars are placed at the center of each bin (`align = "center"`) and the bar `width` is the width of the bin.

You can also add a vertical line to the graph with `ax.axvline`. In the graph below, there is a vertical line added for the 50th percentile that was calculated above. 

The same `bins` should be used for every distribution that you want to compare, so that the bars are directly comparable.

In [ ]:
## bin edges: 0, 5, 10, ..., 150 (all FWI values in the data are below 150)
bins = np.arange(0, 155, 5)

## count the number of baseline-period days in each bin
counts_early, _ = np.histogram(fwi_early["fwi"], bins = bins)

fig, ax = plt.subplots()

ax.bar(bins[:-1], counts_early, width = 5, align = "center", color = "orange", edgecolor = "gray")
ax.axvline(fwi_early_p50, color = "black", linestyle = "--", label = "50th percentile")

ax.set_xlabel("Fire Weather Index (FWI)")
ax.set_ylabel("Number of days")
ax.set_title("Distribution of Daily FWI, 1970-2004")
ax.legend();

<font color="#0000FF">**1b)** Under the historical planning standard, an extreme fire weather day is one above the 99.5th percentile (the top 0.5% of days) of the baseline period. Calculate this threshold from the baseline-period data (`fwi_early`) and save it as `extreme_threshold`.
</font>

In [ ]:
## add code here


<font color="#0000FF">**1c)** Make a bar chart of the distribution of daily FWI in the recent period (2005 and later). Use the same `bins` as in the example above, and add a vertical line showing the extreme threshold you calculated in 1b.
</font>

In [ ]:
## add code here
## remember to label your graph



**Counting extreme fire weather days in a year**

You will notice that the overall shape of the histograms looks pretty similar across the two time periods. You will need to do some additional calculations to look more closesly at what is happening with extreme fire weather days. 

To do this, you will calculate the number of extreme fire weather days in each year. An example of this calculation for 1970 is given in the cell below. The code does the following: 
* First, create a subset of the data for a single year.
* Compare each daily FWI value to the threshold. Python returns `True` if the FWI is above the threshold and `False` if it is not. Python treats `True` as 1 and `False` as 0, so taking the `sum` gives the number of days above the threshold.

In [ ]:
fwi_1970 = fwi.loc[fwi["year"] == 1970]
n_extreme_1970 = sum(fwi_1970["fwi"] > extreme_threshold)
print("Number of extreme fire weather days in 1970:", n_extreme_1970)

<font color="#0000FF">**1d)** Now, calculate the number of extreme fire weather days (those above the extreme threshold) in each year of the data from 1970-2025. 
</font>

Use the following code, adding lines where necessary. The loop goes through each year, and the current year is stored as `yr`. The `enumerate()` function creates a counter variable, `i`, which keeps track of the row of the `annual_freq` DataFrame in which to store the results for each year.

In [ ]:
annual_freq = pd.DataFrame(columns = ["year", "extreme_fwi_days"], index = range(56))

for i, yr in enumerate(range(1970, 2026)):
    annual_freq.loc[i, "year"] = yr ## save the year in the summary table

    # add code to create a subset of the fwi data for the current year

    # add code to count how many days exceeded the extreme threshold, and save it in annual_freq.loc[i, "extreme_fwi_days"]


<font color="#0000FF">Print out the number of extreme fire weather days in 1970 and 2025:
</font>

In [ ]:
## add code here


<font color="#0000FF">**1e)** Create a time series plot showing the number of extreme fire weather days in each year.
</font>

In [ ]:
fig, ax = plt.subplots()

## add code here

## remember to label your graph


<font color="#0000FF">**1f)** Calculate the average number of extreme fire weather days per year in the baseline period (1970-2004) and in the recent period (2005-2025).
</font>

In [ ]:
## add code here


<font color="#0000FF">**1g)** By definition, the planning standard was designed so that extreme conditions that occur on 0.5% of days, or about 1.8 days per year. Based on your results from 1g, how much more frequent were extreme fire weather days in the recent period compared to the baseline period?
</font>

In [ ]:
## add code here


<font color="#0000FF">**1h)** What year had the maximum number of extreme fire weather days? How many days occurred in that year?
</font>

In [ ]:
## add code here


<font color="#0000FF">**1i)** Discussion: Search for information about wildfires near Fort Collins during the year you found in 1h, including any fires near Fort Collins. Briefly describe any major fires you learn about (where, when, and how large) and the conditions that contributed to them. Based on what you find, does FWI appear to be a reliable indicator of fire risk in this region? Why or why not?
</font>

*add your answer here*

<font color="#0000FF">**1j)** Discussion: Summarize the results of your analyses for the rest of the wildfire mitigation team. Is the frequency of extreme fire weather changing in Fort Collins? By how much?
</font>

*add your answer here*

<font color="#0000FF">**1k)** Discussion: In addition to climate change, explain two other reasons why humans have likely impacted wildfire risk in the areas surrounding Fort Collins.

*add your answer here*

### Exporting this notebook as an HTML file

When you're done, convert your work to an HTML file using the code below. The HTML file will preserve your code, text, and output as it appears. You can then submit the HTML file on Canvas.

Before running it, make sure you have run all of the cells you want included so their output shows up. The easiest way is to select "Restart session and run all" under the Runtime dropdown menu at the top of Colab. When it finishes in Colab, the HTML file will download automatically.

In [ ]:
# 1. make sure this matches your notebook's filename (shown at the top-left of the Colab window)
notebook_filename = "Lab4_Extreme_Fire_Weather.ipynb"

# 2. if running in Google Colab, the .ipynb file is not on the runtime's disk, so we
#    grab the current notebook from Colab and save it before converting
try:
    import json
    from google.colab import _message
    notebook_json = _message.blocking_request("get_ipynb", request="", timeout_sec=60)["ipynb"]
    with open(notebook_filename, "w") as f:
        json.dump(notebook_json, f)
    in_colab = True
except ImportError:
    in_colab = False

# 3. convert the notebook to an HTML file
!jupyter nbconvert --to html "{notebook_filename}"

# 4. if running in Google Colab, automatically download the HTML file
if in_colab:
    from google.colab import files
    files.download(notebook_filename.replace(".ipynb", ".html"))